### Unsupervised Learning Outline

Unsupervised Pipeline Step - Description

#### Phase 0: Data Integrity & Validation
Key Question - how does the data look? 

1. Combine CXR meta and chexpert files, filter on pneumonia (0,1) and jpeg image position (Chest PA and Lat)
2. Check for missing/null values, image paths, etc. 

#### Phase 1 - Preprocessing & KMeans Clustering Run (centroid based, spherical shapes)
Key Question - does K-means do a good job of clustering pneumonia vs. not? 

1. Using HuggingFace transform and pre-trained ResNet50: https://huggingface.co/Lab-Rasool/RadImageNet 
2. Extract embeddings from ResNet50 for each image to use for unsupervised task instead of using raw pixels
3. Standardize embeddings using StandardScaler and use PCA for dimensionality reduction fixed to 50 components to create a stable baseline. 
4. Use PCA space for downstream KMeans, DBSCAN and visualizations
5. Conduct a KMeans sweep across a range of predefined clusters from 2 to 10 and evaluate model for silhouette score, ARI and NMI. Goal to find high silhouette and high ARI/NMI.
6. Visualize using tSNE computed from PCA features and compare clusters with pneumonia classification.

#### Phase 2 - Preprocessing & DBSCAN Clustering Run (density based, complex shapes, noise/outlier discovery)
Key Question - are there complex clusters that DBSCAN can identify for pneumonia vs. not? 

1. PCA features same as above for KMeans
2. Optimize parameters: eps (using k-distance plot) and min-samples over a range (5, 10, 20)
3. Evalute how much was assigned noise/outliers and if they are not pneumonia.

#### Phase 3 - Pneumonia Phenotypes (future work)
1. Restrict Pneumonia flag = 1 and same steps as above: PCA, KMeans, DBSCAN, Evaluation, Visualization

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

### EDA and Preprocessing

In [2]:
df_meta = pd.read_csv('mimic-cxr-2.0.0-metadata.csv')
df_meta.sample(5)

,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
149618,6b5dd5e5-296f8dd9-90bc4590-1a102d34-1d5cf724,13984946,51078600,CHEST (PA AND LAT),PA,3056,2544,21221104,170723.265,CHEST (PA AND LAT),postero-anterior,Erect
112371,1432f58a-da597991-2fe9d0eb-912ab57e-2e9f8042,12998429,55341712,CHEST (PA AND LAT),PA,3056,2544,21710611,191910.312,CHEST (PA AND LAT),postero-anterior,Erect
281518,268c22a4-5dc6166a-d2996aa0-5cb87476-77a13a6f,17455650,57214742,NaN,PA,2022,1736,21310325,82339.000,CHEST (PA AND LAT),postero-anterior,Erect
230989,12512307-2fb9995a-a074ce97-45e51e6a-2d7c7734,16107040,58012324,CHEST (PA AND LAT),PA,3056,2544,21200911,10603.984,CHEST (PA AND LAT),postero-anterior,Erect
314055,d7e20462-ed3745e1-4961dc8e-319aacfe-54b49954,18312374,56849601,CHEST (PA AND LAT),LATERAL,3056,2544,21290112,121707.296,CHEST (PA AND LAT),lateral,Erect


In [15]:
df_meta['Columns'].mean(),  df_meta['Rows'].mean(),  df_meta['Rows'].mean() * df_meta['Columns'].mean()

(np.float64(2485.837410835035),
 np.float64(2695.528310042163),
 np.float64(6700645.115067747))

In [3]:
df_meta.shape

(377110, 12)

In [3]:
df_meta['ViewPosition'].value_counts()

ViewPosition
AP                147173
PA                 96161
LATERAL            82853
LL                 35133
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1
Name: count, dtype: int64

In [60]:
df_meta['ViewCodeSequence_CodeMeaning'].value_counts()

ViewCodeSequence_CodeMeaning
antero-posterior         146448
postero-anterior          95858
lateral                   82612
left lateral              35033
Erect                       623
left anterior oblique        21
Recumbent                    18
Name: count, dtype: int64

In [5]:
df_chex = pd.read_csv('mimic-cxr-2.0.0-chexpert.csv')
df_chex.sample(5)

,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
7165,10313534,57646021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
212472,19337001,50873846,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,1.0
135888,15942934,57646974,NaN,-1.0,NaN,NaN,-1.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,1.0
38921,11731363,57343819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN
70666,13113283,54583339,1.0,-1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,NaN


In [6]:
df_chex_img_merged = pd.read_csv('chex_img_merged.csv')
df_chex_img_merged.head()

,subject_id,study_id,image_path,pneumonia
0,19932024,54345212,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0
1,19932024,50963033,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0
2,19932024,58958645,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0
3,19056479,58301271,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1
4,19056479,58301271,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1


In [7]:
df_chex_img_merged.shape

(69580, 4)

In [56]:
df_chex_img_merged['image_path'].value_counts()

image_path
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p19/p19932024/s54345212/68ccd799-61e98428-3c7d8e41-6cbc342c-3426519b.jpg    1
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p19/p19932024/s50963033/fc325dc2-bf224206-062f3c9e-47aae515-4ef25cc0.jpg    1
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p19/p19932024/s58958645/f77e46ef-2fa9d49a-1defe019-ac2c199d-7695e244.jpg    1
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p19/p19056479/s58301271/b5781fa6-94b0b4f2-137b9170-7cf06580-02070968.jpg    1
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p19/p19056479/s58301271/fbc0d48f-623a858b-f75355e4-4f4ea5fa-772f6ed2.jpg    1
                                                                                                                                                      ..
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/fi

In [69]:
df_chex_img_merged['image_path'].nunique()

69580

In [70]:
df_chex_img_merged['study_id'].nunique()

40894

In [67]:
df_chex_img_pos = pd.merge(
    df_chex_img_merged,
    df_meta,
    on=['study_id']
)

In [68]:
df_chex_img_pos.shape

(136820, 15)

In [62]:
df_chex_img_pos.sample(5)

,subject_id,study_id,image_path,pneumonia,dicom_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
4864,19415089,55860892,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1,f855ba78-211dfef4-1f25cd82-439a4f51-cb54175d,NaN,PA,2022,1690,21470125,151147.000,CHEST (PA AND LAT),postero-anterior,Erect
90352,16887429,55671526,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,4a3a37f7-159fe41c-24a0ff64-dabfd2e2-16be702b,NaN,LL,2022,1736,21861109,133219.000,CHEST (PA AND LAT),left lateral,Erect
69617,17600272,57659994,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1,a83349dd-2031bb75-a8f45f6f-8e51cb96-f61fe9ba,CHEST (PA AND LAT),LATERAL,3056,2544,21840327,104616.062,CHEST (PA AND LAT),lateral,Erect
3383,19577720,52945976,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1,45fe78d4-6a5b43a3-0627cd80-946b6f27-3df736e4,CHEST (PORTABLE AP),AP,3050,2539,21520218,64912.078,CHEST (PORTABLE AP),antero-posterior,Erect
94978,16124075,54207545,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,94413295-7b49549a-05a85509-17eabf6b-66ab91e5,CHEST (PA AND LAT),NaN,2140,1760,21681208,82644.000,CHEST (PA AND LAT),NaN,NaN


In [63]:
df_chex_img_pos['image_path'].nunique()

69580

In [64]:
df_chex_img_pos['image_path'].value_counts()

image_path
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/094fb954-385a6333-ddf8a40d-d37d7625-2fa9b214.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/04558279-e261eb60-1d774762-d59a8afe-6a08531c.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ab47f66e-28f7d1a5-a9fa95e8-42457b3b-17f8d8a7.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/89f3a6d9-0395560b-3c14dd99-72b3949c-5b7e8b5f.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ba813696-d50c6819-242d926d-4bb1c964-bdd6eeb1.jpg    9
                                                                                                                                                      ..
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/fi

In [10]:
df_chex_img_pos['ProcedureCodeSequence_CodeMeaning'].value_counts()

ProcedureCodeSequence_CodeMeaning
CHEST (PA AND LAT)                       112293
CHEST (PORTABLE AP)                       24038
DX CHEST WITH DECUB                          88
lateral                                      73
postero-anterior                             64
antero-posterior                             61
DX CHEST PORT LINE/TUBE PLCMT 1 EXAM         40
DX CHEST PORTABLE PICC LINE PLACEMENT        32
DX CHEST PORT LINE/TUBE PLCMT 3 EXAMS        27
DX CHEST 2 VIEW PICC LINE PLACEMENT          24
CHEST (SINGLE VIEW)                          22
DX CHEST & RIBS                              20
DX CHEST PORT LINE/TUBE PLCMT 2 EXAMS        16
CHEST (PRE-OP PA & LAT)                      13
CHEST PORT LINE/TUBE PLCT 1 EXAM              4
CHEST PORT LINE PLACEMENT                     4
CHEST PRE-OP                                  1
Name: count, dtype: int64

In [11]:
df_chex_img_pos.shape

(136820, 6)

In [12]:
df_pa_lat = df_chex_img_pos[df_chex_img_pos['ProcedureCodeSequence_CodeMeaning'] == 'CHEST (PA AND LAT)']

In [23]:
df_pa_lat.sample(5)

,subject_id,study_id,image_path,pneumonia,ViewPosition,ProcedureCodeSequence_CodeMeaning
131720,15655633,55921421,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,NaN,CHEST (PA AND LAT)
85886,16971707,54880610,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,PA,CHEST (PA AND LAT)
42908,18263240,55173781,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,AP,CHEST (PA AND LAT)
46337,18002210,59142300,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,0,PA,CHEST (PA AND LAT)
121287,10568523,58303938,/nfs/turbo/si-acastel/mimic-project/data_raw/m...,1,LATERAL,CHEST (PA AND LAT)


In [53]:
df_pa_lat.shape

(112293, 6)

In [54]:
df_pa_lat['image_path'].value_counts()

image_path
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/094fb954-385a6333-ddf8a40d-d37d7625-2fa9b214.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/1d3ca8d1-c7315114-333076e7-a049ca5d-2e27ccfb.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ab47f66e-28f7d1a5-a9fa95e8-42457b3b-17f8d8a7.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/2a78e6f4-f9a7dbad-e5f4b7a7-0cd45a77-c284eec5.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ba813696-d50c6819-242d926d-4bb1c964-bdd6eeb1.jpg    9
                                                                                                                                                      ..
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/fi

In [39]:
df_pa_lat['ProcedureCodeSequence_CodeMeaning'].isna().mean() * 100

np.float64(0.0)

In [45]:
df_pa_lat['pneumonia'].value_counts()

pneumonia
0    75733
1    36560
Name: count, dtype: int64

In [49]:
df_pa_lat['image_path'].value_counts()

image_path
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/094fb954-385a6333-ddf8a40d-d37d7625-2fa9b214.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/1d3ca8d1-c7315114-333076e7-a049ca5d-2e27ccfb.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ab47f66e-28f7d1a5-a9fa95e8-42457b3b-17f8d8a7.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/2a78e6f4-f9a7dbad-e5f4b7a7-0cd45a77-c284eec5.jpg    9
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/files/p14/p14908132/s50022785/ba813696-d50c6819-242d926d-4bb1c964-bdd6eeb1.jpg    9
                                                                                                                                                      ..
/nfs/turbo/si-acastel/mimic-project/data_raw/mimic-cxr-jpg_2_1_0_gcs/fi

### K Means Clustering Model